# 02 — kvpress: Needle-in-a-Haystack with KeyDiffPress

This notebook runs the Needle-in-a-Haystack (NIAH) benchmark using the
[kvpress](https://github.com/NVIDIA/kvpress) library with Qwen3-8B.

We test **KeyDiffPress** — key similarity-based KV cache compression
([arXiv:2504.15364](https://arxiv.org/abs/2504.15364)) — with compression
applied to both **prefill** and **decoding** phases at multiple compression
ratios and needle depths.

The press configuration combines:
- **KeyDiffPress** for prefill — single-pass eviction over the full sequence
- **CompressionRatioDecodingPress(KeyDiffPress)** for decoding — maintains a fixed compression ratio during generation

These are wrapped together using **PrefillDecodingPress**.

The NIAH benchmark inserts a known "needle" sentence into a long haystack of
Paul Graham essays, then asks the model to retrieve it. This tests whether
KV cache compression preserves retrieval ability at different context positions.

**References:**
- G. Kamradt. *Needle in a haystack — pressure testing LLMs.* GitHub, 2023.
- N. F. Liu et al. *Lost in the middle: How language models use long contexts.* TACL, 2024.

Results are saved to `results/kvpress/` for comparison in notebook 05.

## Configuration

In [1]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.0, 0.25, 0.50, 0.75]

NEEDLE_DEPTHS = [0, 25, 50, 75, 100]

MAX_CONTEXT_LENGTHS = [4096, 8192]

PRESS_CONFIGS = {
    "prefill_decoding_keydiff": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "uniform_filtering_keydiff": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=UniformFilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering_zerofill_keydiff": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering_stale_keydiff": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

MAX_NEW_TOKENS = 64

In [2]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using FORK kvpress from /opt/app-root/src/kvpress-fork
  location: /opt/app-root/src/kvpress-fork/kvpress


## 1. Load Model

In [3]:
import torch
from transformers import pipeline, BitsAndBytesConfig
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress, UniformFilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

if vram_gb < 20:
    print("Using 4-bit quantization (VRAM < 20 GB)")
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU: NVIDIA A100-SXM4-40GB — 42.4 GB VRAM
Using Flash Attention 2


Loading checkpoint shards: 100%|██████████| 5/5 [00:04<00:00,  1.10it/s]
Device set to use cuda:0



Model loaded. GPU memory allocated: 16.38 GB


## 2. Load Haystack Dataset

In [4]:
from datasets import load_dataset

haystack_ds = load_dataset("alessiodevoto/paul_graham_essays", split="test")
haystack_df = haystack_ds.to_pandas()

print(f"Haystack dataset loaded: {len(haystack_df)} rows")
print(f"Columns: {list(haystack_df.columns)}")
print(f"Needle: {haystack_df['needle'].iloc[0][:100]}...")
print(f"Question: {haystack_df['question'].iloc[0]}")

Haystack dataset loaded: 1 rows
Columns: ['context', 'needle', 'question', 'answer_prefix', 'max_new_tokens']
Needle: 

Remember, the best thing to do in San Francisco is eat a sandwich and sit in Dolores Park on a sun...
Question: 

Question: Based on the content of the book, what is the best thing to do in San Francisco?


## 3. Run Inference

For each (press_config, compression_ratio, max_context_length, needle_depth)
combination, we insert the needle into the haystack at the specified depth,
run the kvpress pipeline, and record the prediction.

In [5]:
import time
import json
from eval_utils import insert_needle_in_haystack, calculate_niah_metrics, rouge_l_f_scores

all_results = []

for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        press = press_factory(ratio)
        for max_ctx_len in MAX_CONTEXT_LENGTHS:
            niah_df = insert_needle_in_haystack(
                haystack_df, pipe.tokenizer, max_ctx_len, NEEDLE_DEPTHS,
            )

            config_label = f"{press_name} | ratio={ratio} | ctx={max_ctx_len}"
            print(f"\n{'='*60}")
            print(f"Running: {config_label}")
            print(f"{'='*60}")

            for idx, row in niah_df.iterrows():
                torch.cuda.reset_peak_memory_stats()
                start = time.perf_counter()

                output = pipe(
                    row["context"],
                    question=row["question"],
                    answer_prefix=row["answer_prefix"],
                    press=press,
                    max_new_tokens=row["max_new_tokens"] or MAX_NEW_TOKENS,
                    max_context_length=max_ctx_len,
                )
                predicted_answer = output["answer"]

                elapsed = time.perf_counter() - start
                peak_mem = torch.cuda.max_memory_allocated() / 1e9

                result = {
                    "framework": "kvpress",
                    "press": press_name,
                    "compression_ratio": ratio,
                    "max_context_length": max_ctx_len,
                    "needle_depth": row["needle_depth"],
                    "needle": row["needle"],
                    "question": row["question"],
                    "predicted_answer": predicted_answer,
                    "elapsed_sec": round(elapsed, 3),
                    "peak_gpu_mem_gb": round(peak_mem, 3),
                    "task": "needle_in_haystack",
                }
                all_results.append(result)

                print(
                    f"  depth={row['needle_depth']:3d}%  "
                    f"time={elapsed:.1f}s  mem={peak_mem:.2f}GB  "
                    f"answer={predicted_answer[:60]}..."
                )

                torch.cuda.empty_cache()

print(f"\nDone. Total results: {len(all_results)}")

Token indices sequence length is longer than the specified maximum sequence length for this model (699555 > 131072). Running this sequence through the model will result in indexing errors



Running: prefill_decoding_keydiff | ratio=0.0 | ctx=4096
  depth=  0%  time=1.7s  mem=17.40GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 25%  time=1.3s  mem=17.40GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 50%  time=1.3s  mem=17.40GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 75%  time=1.3s  mem=17.40GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth=100%  time=1.3s  mem=17.40GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....

Running: prefill_decoding_keydiff | ratio=0.0 | ctx=8192
  depth=  0%  time=1.8s  mem=18.44GB  answer= **eat a sandwich and sit in Dolores Park on a sunny day**....
  depth= 25%  time=1.8s  mem=18.44GB  answer= **to eat a sandwich and sit in Dolores Park on a sunny day*...
  depth= 50%  time=1.8s  mem=18.44GB  answer= **eat a sandwich and sit in Dolores Park on a sunny day**....
  depth= 75%  time=1.8s 

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Running: prefill_decoding_keydiff | ratio=0.25 | ctx=4096
  depth=  0%  time=1.4s  mem=17.25GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 25%  time=1.3s  mem=17.25GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 50%  time=1.3s  mem=17.25GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 75%  time=1.3s  mem=17.25GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth=100%  time=1.3s  mem=17.25GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....

Running: prefill_decoding_keydiff | ratio=0.25 | ctx=8192
  depth=  0%  time=1.8s  mem=18.14GB  answer= **eat a sandwich and sit in Dolores Park on a sunny day**....
  depth= 25%  time=1.7s  mem=18.14GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 50%  time=1.7s  mem=18.14GB  answer= to eat a sandwich and sit in Dolores Park on a sunny day....
  depth= 75%  time=1.8s  

## 4. Score Predictions

In [6]:
import pandas as pd

df = pd.DataFrame(all_results)

metrics = calculate_niah_metrics(df)
df["rouge_l_f"] = rouge_l_f_scores(metrics)

summary = (
    df.groupby(["press", "compression_ratio", "max_context_length", "needle_depth"])
    .agg(
        rouge_l=("rouge_l_f", "mean"),
        mean_time=("elapsed_sec", "mean"),
        mean_mem=("peak_gpu_mem_gb", "mean"),
    )
    .round(4)
)

print(summary.to_string())

                                                                              rouge_l  mean_time  mean_mem
press                      compression_ratio max_context_length needle_depth                              
filtering_stale_keydiff    0.00              4096               0              0.7429      1.772    17.400
                                                                25             0.7429      1.762    17.400
                                                                50             0.7429      1.763    17.400
                                                                75             0.7429      1.757    17.400
                                                                100            0.7429      1.759    17.400
                                             8192               0              0.7059      2.414    18.442
                                                                25             0.7429      2.481    18.442
                                     

## 5. Save Results

In [7]:
import os

os.makedirs("results/kvpress", exist_ok=True)

predictions_path = "results/kvpress/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")

Saved predictions to results/kvpress/predictions.csv
Saved metrics to results/kvpress/metrics.json
